# Phase 2b - Deep-learning retrain (fixes for the first DL run)

Run this **after** you have finalised the split and feature list from the ML diagnostics
(duplicate check, `Destination Port` ablation). DL training is expensive, so retrain **once**, on the final data.

What changes compared with the first run:
1. `log1p` + `StandardScaler` + clip to +/-5, fitted on train only
2. No class weights and no oversampling - the decision threshold is tuned on validation instead
3. Adam with gradient clipping, `ReduceLROnPlateau`, early stopping on validation PR-AUC
4. New **MLP** (cleanest target for FGSM/PGD in Phase 3), LSTM stabilised
5. Each model is saved as `.keras` + metrics JSON + plots, same schema as your ML results

**Expected inputs** (from your preprocessing notebook): `X_train, X_val, X_test` as pandas DataFrames that are
cleaned but **not scaled**, and `y_train, y_val, y_test` (0 = benign, 1 = attack).

In [1]:
import os, json, time, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve,
                             precision_recall_curve, ConfusionMatrixDisplay)

# ===================== CONFIG (edit here) =====================
OUT_DIR    = "dl_phase2b"      # everything is saved here
SEED       = 42
BATCH      = 1024              # larger batch = faster epochs
EPOCHS     = 40                # early stopping will usually stop sooner
MAX_MIN    = 120               # hard time limit per model (minutes)
TRAIN_FRAC = 1.0               # use 0.2 for a quick smoke test on 20% of train
SEQ_CHUNK  = 1                 # 1 = paper-faithful (one feature per timestep).
                               # Larger values (e.g. 7 or 11 if F=77) make RNNs several times faster.
MODELS     = ["mlp", "cnn", "bilstm", "bigru", "lstm"]
LR         = {"mlp": 1e-3, "cnn": 1e-3, "bilstm": 1e-3, "bigru": 1e-3, "lstm": 3e-4}
# ==============================================================

os.makedirs(OUT_DIR, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print("TensorFlow", tf.__version__, "| GPUs:", gpus)
if not gpus:
    print("WARNING: no GPU found. RNN training will be very slow. "
          "Use Colab/Kaggle (free GPU), WSL2, or raise SEQ_CHUNK.")

ModuleNotFoundError: No module named 'tensorflow'

## Step 1 - Diagnose what the first DL run actually saw

Read the output like this:
- **Many features with max > 1e5** -> the data is still raw, so the scaling in Step 2 will work as intended.
- **Global max <= 1 and many features with p99 < 0.01** -> the data was Min-Max scaled. Go back to your
  preprocessing notebook and export the *unscaled* cleaned features, then rerun from here.
- **Attack share in train much higher than in val/test (for example 0.5 vs 0.17)** -> you trained on rebalanced data.
  That alone explains high recall with low precision. Do not rebalance; tune the threshold instead.

In [ ]:
def describe_raw(X, name):
    s = X.sample(min(200_000, len(X)), random_state=0)
    p99 = s.quantile(0.99)
    print(f"{name}: global min={s.min().min():.3g}   global max={s.max().max():.3g}")
    print(f"  features with max > 1e5 : {int((s.max() > 1e5).sum())} / {s.shape[1]}")
    print(f"  features with p99 < 0.01: {int((p99 < 0.01).sum())} / {s.shape[1]}")

describe_raw(X_train, "X_train")
print("attack share train / val / test:",
      round(float(np.mean(y_train)), 3), round(float(np.mean(y_val)), 3), round(float(np.mean(y_test)), 3))
print("all finite:", bool(np.isfinite(X_train).all().all()))
print("n_features:", X_train.shape[1], "| train/val/test rows:", len(X_train), len(X_val), len(X_test))

## Step 2 - Scale for deep models (fit on train only)

`log1p` tames the heavy-tailed CICIDS2017 columns, `StandardScaler` centres them, and clipping to +/-5 stops
rare extreme flows from blowing up the recurrent layers (a likely cause of the LSTM divergence).
Negative values (such as the `-1` sentinels in window-size features) are mapped to 0 before the log.
The scaler is saved because the adversarial phase needs it to move between raw and scaled space.

In [ ]:
def to_log(X):
    return np.log1p(np.clip(X.to_numpy(dtype="float64"), 0, None))

scaler = StandardScaler().fit(to_log(X_train))            # TRAIN ONLY

def prep(X):
    return np.clip(scaler.transform(to_log(X)), -5, 5).astype("float32")

Xtr, Xva, Xte = prep(X_train), prep(X_val), prep(X_test)
ytr, yva, yte = [np.asarray(v).astype("float32") for v in (y_train, y_val, y_test)]
joblib.dump({"scaler": scaler, "columns": list(X_train.columns)}, f"{OUT_DIR}/scaler_logstd.joblib")

if TRAIN_FRAC < 1:
    Xtr, _, ytr, _ = train_test_split(Xtr, ytr, train_size=TRAIN_FRAC, stratify=ytr, random_state=SEED)

F = Xtr.shape[1]
print("train", Xtr.shape, "| val", Xva.shape, "| test", Xte.shape)
print("scaled range:", float(Xtr.min()), float(Xtr.max()), "| train attack share:", float(ytr.mean()))
print("valid SEQ_CHUNK values:", [d for d in range(1, F + 1) if F % d == 0])

## Step 3 - Model definitions

Every model takes a flat `(n_features,)` input and reshapes internally. This keeps the models directly usable
by the Adversarial Robustness Toolbox in Phase 3. CNN-1D and the RNN sizes follow the reference paper.

In [ ]:
assert F % SEQ_CHUNK == 0, f"SEQ_CHUNK must divide {F}"
STEPS, FEAT = F // SEQ_CHUNK, SEQ_CHUNK

def build(name):
    inp = keras.Input(shape=(F,), name="flows")
    if name == "mlp":
        x = layers.Dense(128, activation="relu")(inp)
        x = layers.BatchNormalization()(x); x = layers.Dropout(0.2)(x)
        x = layers.Dense(64, activation="relu")(x)
        x = layers.BatchNormalization()(x); x = layers.Dropout(0.2)(x)
        x = layers.Dense(32, activation="relu")(x)
    elif name == "cnn":
        x = layers.Reshape((F, 1))(inp)
        x = layers.Conv1D(64, 3, activation="relu")(x); x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(32, 3, activation="relu")(x); x = layers.MaxPooling1D(2)(x)
        x = layers.Flatten()(x)
        x = layers.Dense(64, activation="relu")(x); x = layers.Dropout(0.5)(x)
    elif name in ("lstm", "bilstm", "bigru"):
        x = layers.Reshape((STEPS, FEAT))(inp)
        if name == "lstm":
            x = layers.LSTM(50)(x)
        elif name == "bilstm":
            x = layers.Bidirectional(layers.LSTM(50))(x)
        else:
            x = layers.Bidirectional(layers.GRU(50))(x)
        x = layers.Dropout(0.2)(x)
    else:
        raise ValueError(name)
    out = layers.Dense(1, activation="sigmoid", name="p_attack")(x)
    return keras.Model(inp, out, name=name)

for n in MODELS:
    print(f"{n:7s} params = {build(n).count_params():,}")

## Step 4 - Training and evaluation helpers

Thresholds are chosen on the **validation** set only and then applied unchanged to the test set:
- `thr_0.5` - the default (what your first run reported)
- `thr_best_f1` - maximises F1 on validation
- `thr_fpr_le_1pct` - highest recall while keeping validation false-positive rate at or below 1% (a realistic IDS operating point)

In [ ]:
class TimeLimit(keras.callbacks.Callback):
    def __init__(self, minutes):
        super().__init__(); self.limit = minutes * 60
    def on_train_begin(self, logs=None):
        self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        if time.time() - self.t0 > self.limit:
            print(f"\nTime limit reached after epoch {epoch + 1}")
            self.model.stop_training = True

def fit(name):
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = build(name)
    model.compile(
        optimizer=keras.optimizers.Adam(LR[name], clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc"), keras.metrics.AUC(name="pr_auc", curve="PR")])
    cbs = [
        keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc", mode="max", factor=0.5, patience=2, min_lr=1e-5),
        keras.callbacks.ModelCheckpoint(f"{OUT_DIR}/{name}.keras", monitor="val_pr_auc", mode="max", save_best_only=True),
        keras.callbacks.CSVLogger(f"{OUT_DIR}/{name}_log.csv"),
        TimeLimit(MAX_MIN),
    ]
    t0 = time.time()
    h = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS,
                  batch_size=BATCH, callbacks=cbs, verbose=2)
    hist = {k: [float(v) for v in vs] for k, vs in h.history.items()}
    return model, hist, time.time() - t0

def metrics(y, p, thr):
    y = np.asarray(y).astype(int)
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(threshold=float(thr),
                accuracy=float((tp + tn) / len(y)),
                precision=float(tp / max(tp + fp, 1)),
                recall=float(tp / max(tp + fn, 1)),
                f1=float(2 * tp / max(2 * tp + fp + fn, 1)),
                specificity=float(tn / max(tn + fp, 1)),
                fpr=float(fp / max(fp + tn, 1)),
                fnr=float(fn / max(fn + tp, 1)),
                roc_auc=float(roc_auc_score(y, p)),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def thr_f1(y, p):
    pr, rc, th = precision_recall_curve(np.asarray(y).astype(int), p)
    f1 = 2 * pr * rc / (pr + rc + 1e-12)
    return float(th[np.argmax(f1[:-1])])

def thr_fpr(y, p, max_fpr=0.01):
    fpr, tpr, th = roc_curve(np.asarray(y).astype(int), p)
    return float(th[np.where(fpr <= max_fpr)[0][-1]])

def plot_cm_roc(name, y, p, thr, path):
    y = np.asarray(y).astype(int)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ConfusionMatrixDisplay.from_predictions(y, (p >= thr).astype(int), display_labels=["Benign", "Attack"],
                                            cmap="Blues", values_format="d", ax=ax[0])
    ax[0].set_title(f"{name} - Confusion Matrix (Test, thr={thr:.3f})")
    fpr, tpr, _ = roc_curve(y, p)
    ax[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y, p):.4f}")
    ax[1].plot([0, 1], [0, 1], "--", c="gray")
    ax[1].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve (Test)")
    ax[1].legend(loc="lower right")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.show()

def plot_curves(name, h, path):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(h["loss"], label="train"); ax[0].plot(h["val_loss"], label="validation")
    ax[0].set(title=f"{name} - Loss", xlabel="Epoch"); ax[0].legend()
    ax[1].plot(h["val_pr_auc"]); ax[1].set(title="Validation PR-AUC", xlabel="Epoch")
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.show()

def evaluate(name, model, hist, secs):
    pva = model.predict(Xva, batch_size=8192, verbose=0).ravel()
    pte = model.predict(Xte, batch_size=8192, verbose=0).ravel()
    rules = {"thr_0.5": 0.5,
             "thr_best_f1": thr_f1(yva, pva),
             "thr_fpr_le_1pct": thr_fpr(yva, pva, 0.01)}
    res = {"model": name, "params": int(model.count_params()), "train_time_s": float(secs),
           "epochs_run": len(hist["loss"]),
           "settings": {"batch": BATCH, "epochs": EPOCHS, "lr": LR[name],
                        "seq_chunk": SEQ_CHUNK, "train_frac": TRAIN_FRAC}}
    for rule, t in rules.items():
        res[rule] = {"validation": metrics(yva, pva, t), "test": metrics(yte, pte, t)}
    json.dump(res, open(f"{OUT_DIR}/{name}_metrics.json", "w"), indent=2)
    json.dump(hist, open(f"{OUT_DIR}/{name}_history.json", "w"))
    np.save(f"{OUT_DIR}/{name}_test_probs.npy", pte)
    plot_cm_roc(name, yte, pte, rules["thr_best_f1"], f"{OUT_DIR}/{name}_cm_roc.png")
    plot_curves(name, hist, f"{OUT_DIR}/{name}_curves.png")
    for rule in rules:
        m = res[rule]["test"]
        print(f"{name:7s} {rule:16s} thr={m['threshold']:.3f} acc={m['accuracy']:.4f} "
              f"prec={m['precision']:.4f} rec={m['recall']:.4f} fpr={m['fpr']:.4f} auc={m['roc_auc']:.4f}")

## Step 5 - Train all models

Start with `TRAIN_FRAC = 0.2` and `MODELS = ["mlp"]` as a smoke test (a few minutes), then set `TRAIN_FRAC = 1.0`
and run the full list. Finished models are skipped automatically, so you can interrupt and resume.

In [ ]:
for name in MODELS:
    if os.path.exists(f"{OUT_DIR}/{name}_metrics.json"):
        print(f"[skip] {name} already done"); continue
    print(f"\n===== {name} =====")
    model, hist, secs = fit(name)
    evaluate(name, model, hist, secs)

## Step 6 - Compare

Rule of thumb for picking the Phase 3 attack target: highest `recall` at `thr_fpr_le_1pct`,
with test AUC of about 0.995 or better. Compare against your XGBoost row
(99.88% accuracy, 0.14% FPR) in the report, and say plainly that tree models lead on this tabular data.

In [ ]:
rows = []
for name in MODELS:
    f = f"{OUT_DIR}/{name}_metrics.json"
    if not os.path.exists(f):
        continue
    r = json.load(open(f))
    for rule in ("thr_0.5", "thr_best_f1", "thr_fpr_le_1pct"):
        m = r[rule]["test"]
        rows.append(dict(model=name, rule=rule, thr=m["threshold"], acc=m["accuracy"],
                         precision=m["precision"], recall=m["recall"], f1=m["f1"], fpr=m["fpr"],
                         auc=m["roc_auc"], false_alarms=m["fp"], missed=m["fn"],
                         epochs=r["epochs_run"], train_min=r["train_time_s"] / 60))
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT_DIR}/dl_summary.csv", index=False)
summary.round(4)